In [1]:
!python -V

Python 3.12.14


In [2]:
import pandas as pd

In [3]:
import pickle

In [4]:
import seaborn as sns
import matplotlib.pyplot as plt

In [5]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge

from sklearn.metrics import root_mean_squared_error

In [6]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/home/manwe_acer/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1792773436544, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1792773436544, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [7]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime).dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [8]:
df_train = read_dataframe('./data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('./data/green_tripdata_2021-02.parquet')

In [9]:
len(df_train), len(df_val)

(73908, 61921)

In [10]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [11]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [12]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [13]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

7.7587152044859415

In [14]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [15]:
# pour lasso : passage de int64 à int62
X_train.indices = X_train.indices.astype('int32')                                            
X_train.indptr = X_train.indptr.astype('int32')                                              
                                                                                                 
X_val.indices = X_val.indices.astype('int32')                                                
X_val.indptr = X_val.indptr.astype('int32')                                                  
                                                 

In [16]:
ls = Lasso(0.01)
ls.fit(X_train, y_train)

y_pred = ls.predict(X_val)

root_mean_squared_error(y_val, y_pred)

11.167275942353895

In [17]:
with open('models/lasso.bin', 'wb') as f_out :
    pickle.dump((dv, ls), f_out)

In [31]:
with mlflow.start_run(run_name="lasso test1") : 

    mlflow.set_tag("developper", "BaoLong")
    mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.parquet")
    mlflow.log_param("validation-data-path", "./data/green_tripdata_2021-02.parquet")

    alpha = 0.1
    mlflow.log_param("alpha", alpha)

    ls = Lasso(alpha)
    ls.fit(X_train, y_train)

    y_pred = ls.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open('models/lasso_test1.bin', 'wb') as f_out : 
        pickle.dump((dv, ls), f_out)

    #first way to save a model 
    mlflow.log_artifact(local_path="models/lasso_test1.bin")

In [22]:
import xgboost as xgb

In [23]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [24]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [25]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials()
)

In [28]:
# using the best params we found

params = {
    'learning_rate' : 0.15359873255466536,
    'max_depth' : 23,
    'min_child_weight' : 1.925715942779695,
    'objective' : 'reg:linear',
    'reg_alpha' : 0.050497248848893936,
    'reg_lambda' : 0.018611518693549794,
    'seed' : 42
}

In [ ]:
#instead of using mlflow.start_run, we can use autolog because it support xgboost

mlflow.xgboost.autolog()

booster = xgb.train(
        params=params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
)

2026/10/24 15:20:31 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8bae584b0c5e4d2293d23709557b6628', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current xgboost workflow


/home/manwe_acer/mlops-zoomcamp/02-experiment-tracking/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [15:20:52] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:10.99424
[1]	validation-rmse:10.01946
[2]	validation-rmse:9.24669
[3]	validation-rmse:8.63995
[4]	validation-rmse:8.16715
[5]	validation-rmse:7.80206
[6]	validation-rmse:7.52166
[7]	validation-rmse:7.30500
[8]	validation-rmse:7.13894
[9]	validation-rmse:7.01251
[10]	validation-rmse:6.91494
[11]	validation-rmse:6.83729
[12]	validation-rmse:6.77829
[13]	validation-rmse:6.73216
[14]	validation-rmse:6.69547
[15]	validation-rmse:6.66584
[16]	validation-rmse:6.64203
[17]	validation-rmse:6.62137
[18]	validation-rmse:6.60579
[19]	validation-rmse:6.59245
[20]	validation-rmse:6.58236
[21]	validation-rmse:6.57237
[22]	validation-rmse:6.56423
[23]	validation-rmse:6.55724
[24]	validation-rmse:6.55122
[25]	validation-rmse:6.54704
[26]	validation-rmse:6.54185
[27]	validation-rmse:6.53919
[28]	validation-rmse:6.53621
[29]	validation-rmse:6.53413
[30]	validation-rmse:6.53412
[31]	validation-rmse:6.53194
[32]	validation-rmse:6.53094
[33]	validation-rmse:6.52995
[34]	validation-rmse:6

2026/10/24 15:22:55 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.
2026/10/24 15:22:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/24 15:23:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [35]:
mlflow.xgboost.autolog(disable=True)

In [36]:
with mlflow.start_run():

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, name="models_mlflow")

/home/manwe_acer/mlops-zoomcamp/02-experiment-tracking/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:11:45] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.44174
[1]	validation-rmse:10.76627
[2]	validation-rmse:10.17716
[3]	validation-rmse:9.66439
[4]	validation-rmse:9.21866
[5]	validation-rmse:8.83455
[6]	validation-rmse:8.50406
[7]	validation-rmse:8.22009
[8]	validation-rmse:7.97540
[9]	validation-rmse:7.76692
[10]	validation-rmse:7.58908
[11]	validation-rmse:7.43669
[12]	validation-rmse:7.30705
[13]	validation-rmse:7.19679
[14]	validation-rmse:7.10368
[15]	validation-rmse:7.02341
[16]	validation-rmse:6.95495
[17]	validation-rmse:6.89586
[18]	validation-rmse:6.84619
[19]	validation-rmse:6.80220
[20]	validation-rmse:6.76455
[21]	validation-rmse:6.73297
[22]	validation-rmse:6.70498
[23]	validation-rmse:6.68007
[24]	validation-rmse:6.65944
[25]	validation-rmse:6.64053
[26]	validation-rmse:6.62476
[27]	validation-rmse:6.61039
[28]	validation-rmse:6.59774
[29]	validation-rmse:6.58651
[30]	validation-rmse:6.57663
[31]	validation-rmse:6.56758
[32]	validation-rmse:6.55902
[33]	validation-rmse:6.55052
[34]	validation-rmse:

2026/10/24 16:13:26 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [39]:
model_uri = "runs:/7e06caa2b03a41cf9ff4b55e7d2d2f37/models_mlflow"

loaded_model = mlflow.pyfunc.load_model(model_uri)
loaded_model

mlflow.pyfunc.loaded_model:
  artifact_path: /home/manwe_acer/mlops-zoomcamp/02-experiment-tracking/mlruns/2/models/m-62a86aa967914d66845c750917d868db/artifacts
  flavor: mlflow.xgboost
  run_id: 7e06caa2b03a41cf9ff4b55e7d2d2f37

In [40]:
xgboost_model = mlflow.xgboost.load_model(model_uri)
xgboost_model

In [41]:
y_pred = xgboost_model.predict(valid)

In [43]:
y_pred[:10]

array([14.989175 ,  7.0369577, 16.05526  , 24.238691 ,  9.42884  ,
       17.14162  , 11.674835 ,  8.660516 ,  8.957426 , 20.33289  ],
      dtype=float32)

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", "./data/green_tripdata_2021-01.csv")
        mlflow.log_param("valid-data-path", "./data/green_tripdata_2021-02.csv")
        mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        